# Obtener rutas reales de OSM para el simulador

Este notebook llama a Overpass API para extraer las coordenadas reales de tres rutas:

1. **a6** — Tramo de la A-6 desde Las Rozas hacia Villalba (noroeste).
2. **m505** — Tramo de la M-505 (Carretera de El Escorial) desde Las Rozas hasta San Lorenzo de El Escorial.
3. **cruce-a6-m505** — Ruta combinada que empieza en la A-6 y salta a la M-505.

**Cómo usar:**
1. Ejecuta las celdas en orden (Runtime → Run all, o Shift+Enter una a una).
2. Al final se imprime un bloque de JavaScript listo para pegar en `js/rutas.js`.
3. Copia ese bloque y pásaselo a Claude para que lo integre en el archivo.

**Notas:**
- Overpass es gratis y no requiere API key.
- Las queries son lo suficientemente específicas como para devolver solo los tramos que nos interesan (bounding box + filtro por ref).
- El notebook filtra los nodos a ~12 muestras bien espaciadas por ruta. No queremos cientos de puntos.
- Mirror primario: overpass-api.de. Si falla, hay fallback a kumi.systems.

In [ ]:
# === Imports y configuración ===
import requests
import math
import time
from typing import List, Tuple, Dict, Optional

# User-Agent identificativo, buena práctica con Overpass/OSM
HEADERS = {'User-Agent': 'panel-viaje-dev/1.0 (tragabytes.github.io)'}

# Mirrors de Overpass en orden de preferencia
MIRRORS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.private.coffee/api/interpreter',
]

# Muestras objetivo por ruta (submuestreo)
MUESTRAS_OBJETIVO = 12

# Timeout por petición
TIMEOUT_S = 60

print('Configuración lista.')

Configuración lista.


In [ ]:
# === Helpers ===

def distancia_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Distancia haversine entre dos puntos en metros."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def consultar_overpass(query: str) -> Optional[dict]:
    """Prueba los mirrors en orden hasta que uno responda. Devuelve el JSON parseado."""
    for i, url in enumerate(MIRRORS):
        try:
            print(f'  [{i+1}/{len(MIRRORS)}] Probando {url}…', end=' ')
            t0 = time.time()
            r = requests.post(url, data={'data': query}, headers=HEADERS, timeout=TIMEOUT_S)
            dt = int((time.time() - t0) * 1000)
            if r.status_code == 200:
                data = r.json()
                n_elem = len(data.get('elements', []))
                print(f'OK {dt} ms, {n_elem} elementos')
                return data
            else:
                print(f'HTTP {r.status_code}')
        except Exception as e:
            print(f'ERROR: {type(e).__name__}: {e}')
    return None


def extraer_nodos_ordenados(data: dict) -> List[Tuple[float, float]]:
    """De una respuesta Overpass con ways + nodos (usando 'out geom;'),
    extrae la geometría en orden. Devuelve lista de (lat, lon).

    Usamos 'out geom;' porque incluye directamente la geometría de cada way
    sin tener que hacer joins manuales con los nodos. Más simple y fiable."""
    puntos = []
    for elem in data.get('elements', []):
        if elem.get('type') == 'way' and 'geometry' in elem:
            for g in elem['geometry']:
                puntos.append((g['lat'], g['lon']))
    return puntos


def filtrar_duplicados_consecutivos(puntos: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
    """Quita puntos que son iguales al anterior (comunes en juntas de ways)."""
    if not puntos:
        return []
    out = [puntos[0]]
    for p in puntos[1:]:
        if p != out[-1]:
            out.append(p)
    return out


def ordenar_por_proximidad(puntos: List[Tuple[float, float]], inicio: Tuple[float, float]) -> List[Tuple[float, float]]:
    """Ordena una lista de puntos como una ruta empezando en 'inicio', usando vecino más cercano.

    Esto es necesario porque Overpass puede devolver las ways en cualquier orden, y una carretera
    está formada por muchas ways (cada tramo entre intersecciones). Sin ordenar, el simulador
    saltaría entre tramos de forma no contigua."""
    if not puntos:
        return []
    restantes = list(puntos)
    # Elegir el punto más cercano al 'inicio' como primero
    actual = min(restantes, key=lambda p: distancia_m(p[0], p[1], inicio[0], inicio[1]))
    restantes.remove(actual)
    ruta = [actual]
    while restantes:
        siguiente = min(restantes, key=lambda p: distancia_m(p[0], p[1], actual[0], actual[1]))
        ruta.append(siguiente)
        restantes.remove(siguiente)
        actual = siguiente
    return ruta


def submuestrear_por_distancia(puntos: List[Tuple[float, float]], n_objetivo: int) -> List[Tuple[float, float]]:
    """Coge N puntos bien espaciados por distancia acumulada (no por índice).

    Si tomáramos 1 de cada M por índice, las zonas con nodos densos (curvas)
    quedarían sobrerrepresentadas. Distribuir por distancia acumulada reparte
    los puntos de forma uniforme a lo largo del trazado real."""
    if len(puntos) <= n_objetivo:
        return puntos
    # Acumulados
    dists = [0.0]
    for i in range(1, len(puntos)):
        d = distancia_m(puntos[i-1][0], puntos[i-1][1], puntos[i][0], puntos[i][1])
        dists.append(dists[-1] + d)
    total = dists[-1]
    # Objetivos equidistantes
    objetivos = [total * i / (n_objetivo - 1) for i in range(n_objetivo)]
    resultado = []
    j = 0
    for obj in objetivos:
        while j < len(dists) - 1 and dists[j] < obj:
            j += 1
        resultado.append(puntos[j])
    # Quitar duplicados consecutivos que pueden aparecer si hay picos
    return filtrar_duplicados_consecutivos(resultado)


def resumen_ruta(nombre: str, puntos: List[Tuple[float, float]]):
    """Imprime un resumen de la ruta: número de puntos, distancia total, punto inicial y final."""
    if not puntos:
        print(f'  {nombre}: SIN PUNTOS')
        return
    total = 0.0
    for i in range(1, len(puntos)):
        total += distancia_m(puntos[i-1][0], puntos[i-1][1], puntos[i][0], puntos[i][1])
    print(f'  {nombre}: {len(puntos)} puntos, {total/1000:.1f} km totales')
    print(f'    inicio: {puntos[0][0]:.5f}, {puntos[0][1]:.5f}')
    print(f'    fin:    {puntos[-1][0]:.5f}, {puntos[-1][1]:.5f}')


print('Helpers listos.')

Helpers listos.


## Ruta 1 — A-6 desde Las Rozas hacia Villalba

Filtramos las ways con `ref="A-6"` dentro de un bounding box que cubre desde Las Rozas hasta un poco después de Villalba. Usamos `out geom;` para obtener directamente la geometría ordenada de cada way.

**Bounding box:** sur de Las Rozas (40.475, -3.895) a norte de Villalba (40.580, -4.000).

In [ ]:
query_a6 = """
[out:json][timeout:50];
(
  way["ref"="A-6"]["highway"~"^(motorway|trunk|primary)$"](40.475,-4.000,40.580,-3.895);
);
out geom;
"""

print('Consultando A-6…')
data_a6 = consultar_overpass(query_a6)

if data_a6 is None:
    print('❌ No se pudo obtener la A-6 de ningún mirror.')
    ruta_a6 = []
else:
    crudos = extraer_nodos_ordenados(data_a6)
    print(f'  Nodos crudos: {len(crudos)}')
    limpios = filtrar_duplicados_consecutivos(crudos)
    print(f'  Sin duplicados: {len(limpios)}')
    # Ordenar empezando desde el punto más al sureste (Las Rozas)
    inicio_objetivo = (40.492, -3.882)  # aprox Las Rozas norte
    ordenados = ordenar_por_proximidad(limpios, inicio_objetivo)
    ruta_a6 = submuestrear_por_distancia(ordenados, MUESTRAS_OBJETIVO)
    resumen_ruta('A-6', ruta_a6)

Consultando A-6…
  [1/3] Probando https://overpass-api.de/api/interpreter… OK 582 ms, 26 elementos
  Nodos crudos: 163
  Sin duplicados: 162
  A-6: 12 puntos, 7.3 km totales
    inicio: 40.54434, -3.89385
    fin:    40.58731, -3.95384


## Ruta 2 — M-505 (Carretera de El Escorial)

Filtramos ways con `ref="M-505"` en un bounding box amplio que cubre desde Majadahonda/Las Rozas hasta San Lorenzo de El Escorial.

**Bounding box:** de Las Rozas (40.470, -3.900) a El Escorial (40.620, -4.160).

In [ ]:
query_m505 = """
[out:json][timeout:50];
(
  way["ref"="M-505"](40.470,-4.160,40.620,-3.880);
);
out geom;
"""

print('Consultando M-505…')
data_m505 = consultar_overpass(query_m505)

if data_m505 is None:
    print('❌ No se pudo obtener la M-505 de ningún mirror.')
    ruta_m505 = []
else:
    crudos = extraer_nodos_ordenados(data_m505)
    print(f'  Nodos crudos: {len(crudos)}')
    limpios = filtrar_duplicados_consecutivos(crudos)
    print(f'  Sin duplicados: {len(limpios)}')
    # Ordenar empezando desde el este (Las Rozas) hacia el oeste (El Escorial)
    inicio_objetivo = (40.485, -3.890)
    ordenados = ordenar_por_proximidad(limpios, inicio_objetivo)
    ruta_m505 = submuestrear_por_distancia(ordenados, MUESTRAS_OBJETIVO)
    resumen_ruta('M-505', ruta_m505)

Consultando M-505…
  [1/3] Probando https://overpass-api.de/api/interpreter… HTTP 504
  [2/3] Probando https://overpass.kumi.systems/api/interpreter… OK 1452 ms, 255 elementos
  Nodos crudos: 1434
  Sin duplicados: 1405
  M-505: 11 puntos, 48.7 km totales
    inicio: 40.49407, -3.88353
    fin:    40.48846, -3.87173


## Ruta 3 — cruce-a6-m505

Esta ruta es simplemente la primera mitad de la A-6 seguida de la mitad más "oeste" de la M-505. Nos ahorra otra query.

El objetivo es ver el cambio de pastilla azul (A-6) a granate (M-505) en vivo en el simulador.

In [ ]:
# Primera mitad de A-6 + segunda mitad de M-505
if ruta_a6 and ruta_m505:
    mitad_a6 = ruta_a6[:len(ruta_a6)//2]
    mitad_m505 = ruta_m505[len(ruta_m505)//2:]
    ruta_cruce = mitad_a6 + mitad_m505
    # Submuestreamos de nuevo para que quede en 12 puntos homogéneos
    ruta_cruce = submuestrear_por_distancia(ruta_cruce, MUESTRAS_OBJETIVO)
    resumen_ruta('cruce-a6-m505', ruta_cruce)
else:
    print('❌ No se puede construir el cruce: falta alguna de las rutas base.')
    ruta_cruce = []

  cruce-a6-m505: 12 puntos, 45.8 km totales
    inicio: 40.54434, -3.89385
    fin:    40.48846, -3.87173


## Generar bloque JavaScript para pegar en `js/rutas.js`

La siguiente celda imprime las 3 rutas en formato JavaScript listo para copiar directamente al archivo `rutas.js`. **Copia el bloque completo y pégaselo a Claude** — él se encargará de sustituir las rutas correspondientes manteniendo la estructura del archivo.

In [ ]:
def formatear_ruta_js(nombre: str, descripcion: str, puntos: List[Tuple[float, float]]) -> str:
    """Formatea una ruta como el bloque JS de rutas.js."""
    if not puntos:
        return f"    // {nombre}: SIN DATOS\n"
    lineas = []
    lineas.append(f"    '{nombre}': {{")
    lineas.append(f"      descripcion: '{descripcion}',")
    lineas.append(f"      puntos: [")
    # Calcular distancia acumulada para el comentario
    acum = 0.0
    for i, (lat, lon) in enumerate(puntos):
        if i > 0:
            acum += distancia_m(puntos[i-1][0], puntos[i-1][1], lat, lon)
        km = acum / 1000
        nota = f'punto {i+1}/{len(puntos)} · km {km:.1f} · extraído de OSM'
        coma = ',' if i < len(puntos) - 1 else ''
        lineas.append(f"        {{ lat: {lat:.5f}, lon: {lon:.5f}, nota: '{nota}' }}{coma}")
    lineas.append(f"      ]")
    lineas.append(f"    }}")
    return '\n'.join(lineas)


print('=' * 72)
print('COPIAR DESDE AQUÍ Y PEGAR EN EL CHAT')
print('=' * 72)
print()
print('// --- Bloque generado por obtener_rutas_osm.ipynb ---')
print(f'// Fecha: (rellenar)')
print(f'// Muestras objetivo por ruta: {MUESTRAS_OBJETIVO}')
print()
print(formatear_ruta_js('a6', 'Tramo de la A-6 saliendo de Las Rozas hacia Villalba (datos OSM)', ruta_a6))
print(',')
print()
print(formatear_ruta_js('m505', 'Carretera de El Escorial (M-505) desde Las Rozas hasta El Escorial (datos OSM)', ruta_m505))
print(',')
print()
print(formatear_ruta_js('cruce-a6-m505', 'Mitad A-6 + mitad M-505 para probar cambio azul→granate (datos OSM)', ruta_cruce))
print()
print('=' * 72)
print('HASTA AQUÍ')
print('=' * 72)

COPIAR DESDE AQUÍ Y PEGAR EN EL CHAT

// --- Bloque generado por obtener_rutas_osm.ipynb ---
// Fecha: (rellenar)
// Muestras objetivo por ruta: 12

    'a6': {
      descripcion: 'Tramo de la A-6 saliendo de Las Rozas hacia Villalba (datos OSM)',
      puntos: [
        { lat: 40.54434, lon: -3.89385, nota: 'punto 1/12 · km 0.0 · extraído de OSM' },
        { lat: 40.55086, lon: -3.89703, nota: 'punto 2/12 · km 0.8 · extraído de OSM' },
        { lat: 40.55640, lon: -3.90011, nota: 'punto 3/12 · km 1.4 · extraído de OSM' },
        { lat: 40.56152, lon: -3.90299, nota: 'punto 4/12 · km 2.1 · extraído de OSM' },
        { lat: 40.56521, lon: -3.91019, nota: 'punto 5/12 · km 2.8 · extraído de OSM' },
        { lat: 40.56863, lon: -3.91735, nota: 'punto 6/12 · km 3.5 · extraído de OSM' },
        { lat: 40.57219, lon: -3.92406, nota: 'punto 7/12 · km 4.2 · extraído de OSM' },
        { lat: 40.57433, lon: -3.93005, nota: 'punto 8/12 · km 4.8 · extraído de OSM' },
        { lat: 40.57734,

## Verificación visual rápida (opcional)

Si quieres comprobar que los puntos caen donde deben antes de pasárselos a Claude, ejecuta la siguiente celda. Abrirá un enlace a Google Maps con todos los puntos de cada ruta marcados. Si el trazado sigue la carretera real, las coordenadas están bien.

In [ ]:
def enlace_gmaps(puntos: List[Tuple[float, float]]) -> str:
    """Genera un enlace a Google Maps mostrando los puntos como marcadores.
    Formato: https://www.google.com/maps/dir/lat1,lon1/lat2,lon2/...
    Esto abre una ruta 'driving' que conecta los puntos, así que visualmente
    se ve si siguen la carretera o no."""
    if not puntos:
        return '(sin puntos)'
    partes = [f'{lat:.5f},{lon:.5f}' for lat, lon in puntos]
    return 'https://www.google.com/maps/dir/' + '/'.join(partes)


print('Enlaces de verificación en Google Maps:')
print()
print(f'A-6:')
print(f'  {enlace_gmaps(ruta_a6)}')
print()
print(f'M-505:')
print(f'  {enlace_gmaps(ruta_m505)}')
print()
print(f'cruce-a6-m505:')
print(f'  {enlace_gmaps(ruta_cruce)}')

Enlaces de verificación en Google Maps:

A-6:
  https://www.google.com/maps/dir/40.54434,-3.89385/40.55086,-3.89703/40.55640,-3.90011/40.56152,-3.90299/40.56521,-3.91019/40.56863,-3.91735/40.57219,-3.92406/40.57433,-3.93005/40.57734,-3.93682/40.57857,-3.94352/40.58217,-3.94889/40.58731,-3.95384

M-505:
  https://www.google.com/maps/dir/40.49407,-3.88353/40.51663,-3.93492/40.54557,-3.95916/40.57202,-3.99840/40.57802,-4.06101/40.57947,-4.12145/40.57874,-4.04200/40.56869,-3.97820/40.52999,-3.94252/40.49274,-3.88097/40.48846,-3.87173

cruce-a6-m505:
  https://www.google.com/maps/dir/40.54434,-3.89385/40.55086,-3.89703/40.55640,-3.90011/40.56152,-3.90299/40.56521,-3.91019/40.56863,-3.91735/40.57947,-4.12145/40.57874,-4.04200/40.56869,-3.97820/40.52999,-3.94252/40.49274,-3.88097/40.48846,-3.87173
